In [ ]:
# Import necessary libraries
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.models import Sequential
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Load and preprocess the dataset
file_path = 'filtered_acled_20241116_1640.csv'  # Adjust path as needed
data = pd.read_csv('/content/filtered_acled_20241116_1640.csv')

# Select relevant columns for prediction
data_filtered = data[['latitude', 'longitude', 'fatalities']].dropna()

# Convert latitude, longitude, and fatalities to numeric and scale appropriately
data_filtered['latitude'] = pd.to_numeric(data_filtered['latitude'], errors='coerce')
data_filtered['longitude'] = pd.to_numeric(data_filtered['longitude'], errors='coerce')
data_filtered['fatalities'] = pd.to_numeric(data_filtered['fatalities'], errors='coerce')

# Drop rows with any NaN values after conversion
data_filtered = data_filtered.dropna()

# Normalize data for better model performance
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data_filtered)

# Prepare sequences for LSTM input
sequence_length = 5  # Define the sequence length
features = data_scaled

X, y = [], []
for i in range(len(features) - sequence_length):
    X.append(features[i:i + sequence_length])  # Append sequences of data
    y.append(features[i + sequence_length])  # Target is the next row

X = np.array(X)
y = np.array(y)

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Custom callback to print validation accuracy
class EpochMetricsCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        accuracy = 1 - logs['val_loss']  # Custom "accuracy" based on loss
        print(f"Epoch {epoch + 1}: Validation Accuracy: {accuracy:.4f}")

# Define an optimized RNN model
model = Sequential([
    # First LSTM layer with Batch Normalization and Dropout
    LSTM(128, activation='tanh', return_sequences=True, input_shape=(sequence_length, 3)),
    BatchNormalization(),
    Dropout(0.3),

    # Second LSTM layer with Batch Normalization and Dropout
    LSTM(64, activation='tanh', return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),

    # Third LSTM layer with Batch Normalization and Dropout
    LSTM(32, activation='tanh', return_sequences=False),
    BatchNormalization(),
    Dropout(0.2),

    # Dense output layer
    Dense(3)  # Outputs for fatalities, latitude, longitude
])

# Compile the model with an optimized learning rate and optimizer
optimizer = Adam(learning_rate=0.001)  # Start with a smaller learning rate
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Add callbacks for better optimization
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6
)
early_stopping = EarlyStopping(
    monitor='val_loss', patience=10, verbose=1, restore_best_weights=True
)
checkpoint = ModelCheckpoint(
    'best_model.keras', monitor='val_loss', save_best_only=True, verbose=1
)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,  # Increase epochs to give the model more time to converge
    batch_size=64,  # Increase batch size for faster training and more stable gradients
    callbacks=[reduce_lr, early_stopping, checkpoint, EpochMetricsCallback()],
    verbose=1
)

# Save the final model
model.save('rnn_fatalities_forecast.keras')


FileNotFoundError: [Errno 2] No such file or directory: '/content/filtered_acled_20241116_1640.csv'

In [ ]:
from tensorflow.keras.models import load_model

# Load the saved model
model_file = "best_model.keras"  # Path to the saved model
model = load_model(model_file)
print("Model loaded successfully.")

Model loaded successfully.


In [ ]:
import numpy as np
import pandas as pd
import joblib

# Load the scaler used during training
scaler = joblib.load('scaler.pkl')

# Load the most recent data
recent_data = pd.read_csv("/content/filtered_acled_20241116_1640.csv")  # Replace with your actual data source

# Select relevant columns (latitude, longitude, fatalities)
recent_data = recent_data[['latitude', 'longitude', 'fatalities']]

# Scale the data
scaled_recent_data = scaler.transform(recent_data)

# Create sequences
sequence_length = 5  # Use the same sequence length as in training
X_input = np.array([scaled_recent_data[-sequence_length:]])  # Last 5 rows
print(f"Input shape for prediction: {X_input.shape}")


Input shape for prediction: (1, 5, 3)


In [ ]:
# Predict the next step (e.g., fatalities, latitude, longitude for the next day)
predictions = model.predict(X_input)

# Reverse scaling to get predictions in original units
predicted_values = scaler.inverse_transform(predictions)
print(f"Predicted values (fatalities, latitude, longitude): {predicted_values}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step
Predicted values (fatalities, latitude, longitude): [[49.033566   34.841923    0.96976894]]


In [ ]:
future_predictions = []
current_input = X_input.copy()  # Start with the last 5 days of data

for _ in range(30):  # Predict for the next 30 days
    pred = model.predict(current_input)
    pred_rescaled = scaler.inverse_transform(pred)  # Reverse scaling
    future_predictions.append(pred_rescaled[0])  # Add to predictions

    # Update input sequence: Remove oldest data, append the new prediction
    new_row = pred  # Scaled prediction
    current_input = np.append(current_input[:, 1:, :], [new_row], axis=1)

# Convert predictions to a DataFrame for better visualization
future_predictions_df = pd.DataFrame(future_predictions, columns=['fatalities', 'latitude', 'longitude'])
print(future_predictions_df)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━

In [ ]:
# Save predictions to CSV
future_predictions_df.to_csv("future_predictions.csv", index=False)

# Print or plot predictions
print(future_predictions_df)


    fatalities   latitude  longitude
0    49.033566  34.841923   0.969769
1    48.840954  33.756973   0.952732
2    48.724190  33.117252   0.939829
3    48.586166  32.619827   0.928256
4    48.386868  32.206970   0.917380
5    48.277950  32.034538   0.912004
6    48.143909  31.855297   0.906139
7    48.030018  31.707283   0.901591
8    47.924114  31.580193   0.897726
9    47.825771  31.465826   0.894364
10   47.741585  31.366638   0.891505
11   47.662468  31.276260   0.888874
12   47.590469  31.195560   0.886527
13   47.524029  31.122923   0.884404
14   47.462715  31.057545   0.882482
15   47.406399  30.998838   0.880744
16   47.354134  30.945576   0.879155
17   47.305695  30.897232   0.877703
18   47.260658  30.853163   0.876372
19   47.218735  30.812902   0.875148
20   47.179668  30.776011   0.874019
21   47.143177  30.742102   0.872975
22   47.109051  30.710854   0.872008
23   47.077087  30.682003   0.871109
24   47.047100  30.655312   0.870273
25   47.018951  30.630575   0.869493
2